In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/Pakistan Largest Ecommerce Dataset.csv",
    low_memory=False
)

print("Original shape:", df.shape)
df.head()


In [ ]:
df = df.dropna(how="all")

df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

print("Shape after removing empty rows/columns:", df.shape)
print(df.columns.tolist())


In [ ]:
df.info()


In [ ]:
df.isnull().sum().sort_values(ascending=False)


In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

columns_to_drop = [
    "sales_commission_code",
    "working_date",
    "bi_status",
    "mv",
    "year",
    "month",
    "m-y",
    "fy"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

print(df.shape)
print(df.columns.tolist())


In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

print("Date dtype:", df["created_at"].dtype)
print("Missing dates:", df["created_at"].isna().sum())
print("Earliest date:", df["created_at"].min())
print("Latest date:", df["created_at"].max())


In [ ]:
print("Missing values before handling:")
display(df.isnull().sum().sort_values(ascending=False))

df = df.dropna(subset=["status"]).copy()

df["sku"] = df["sku"].fillna("Unknown")
df["category_name_1"] = df["category_name_1"].fillna("Unknown")

print("\nMissing values after handling:")
display(df.isnull().sum().sort_values(ascending=False))


In [ ]:
status_counts = df["status"].value_counts()
status_percent = df["status"].value_counts(normalize=True).mul(100)

display(status_counts)
display(status_percent)


In [ ]:
successful_statuses = [
    "complete",
    "received",
    "paid",
    "closed"
]

sales_df = df[df["status"].isin(successful_statuses)].copy()

print("All cleaned transactions:", len(df))
print("Successful sales item rows:", len(sales_df))
display(sales_df["status"].value_counts())


In [ ]:
print("Successful item rows:", len(sales_df))
print("Unique successful orders:", sales_df["increment_id"].nunique())

display(sales_df["increment_id"].value_counts().head(10))


In [ ]:
sample_order = sales_df["increment_id"].value_counts().idxmax()

display(
    sales_df[
        sales_df["increment_id"] == sample_order
    ][[
        "increment_id",
        "sku",
        "price",
        "qty_ordered",
        "grand_total",
        "discount_amount"
    ]]
)


In [ ]:
orders_df = (
    sales_df
    .sort_values("created_at")
    .drop_duplicates(subset="increment_id", keep="first")
    .copy()
)

print("Successful item rows:", len(sales_df))
print("Unique successful orders:", len(orders_df))
print("Total realized revenue:", orders_df["grand_total"].sum())
print("Average order value:", orders_df["grand_total"].mean())


In [ ]:
display(orders_df["grand_total"].describe())

print("Zero totals:", (orders_df["grand_total"] == 0).sum())
print("Negative totals:", (orders_df["grand_total"] < 0).sum())

display(
    orders_df.nlargest(10, "grand_total")[
        ["increment_id", "created_at", "grand_total", "status"]
    ]
)


In [ ]:
outlier_order = "100323649"

outlier_items = sales_df[
    sales_df["increment_id"] == outlier_order
].copy()

outlier_items["item_value"] = (
    outlier_items["price"] * outlier_items["qty_ordered"]
)

display(
    outlier_items[[
        "increment_id",
        "sku",
        "price",
        "qty_ordered",
        "grand_total",
        "discount_amount",
        "category_name_1"
    ]]
)

print("Calculated item total:", outlier_items["item_value"].sum())
print("Reported grand total:", outlier_items["grand_total"].iloc[0])


In [ ]:
revenue_orders_df = orders_df[
    orders_df["grand_total"] > 0
].copy()

print("Orders before:", len(orders_df))
print("Revenue-generating orders:", len(revenue_orders_df))
print("Removed:", len(orders_df) - len(revenue_orders_df))


In [ ]:
daily_sales = (
    revenue_orders_df
    .groupby("created_at")
    .agg(
        revenue=("grand_total", "sum"),
        orders=("increment_id", "nunique")
    )
    .reset_index()
    .rename(columns={"created_at": "date"})
)

display(daily_sales.head())

print("Daily dataset shape:", daily_sales.shape)
print("First date:", daily_sales["date"].min())
print("Last date:", daily_sales["date"].max())

display(daily_sales.describe())


In [ ]:
full_date_range = pd.date_range(
    start=daily_sales["date"].min(),
    end=daily_sales["date"].max(),
    freq="D"
)

missing_dates = full_date_range.difference(daily_sales["date"])

print("Expected number of days:", len(full_date_range))
print("Actual number of days:", len(daily_sales))
print("Missing dates:", len(missing_dates))
print(missing_dates[:20])
